# Day 2-04｜YOLO Detection、即時場地 Keypoint 與 Homography 整合
> Python 籃球運動資料分析課程  
> 本單元結合 YOLO detector、即時場地 keypoint 與 Homography 整合流程，將球員位置持續投影到 BEV，對實際影片輸出左右並排的分析影片。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 使用 YOLO detector 取得 player bbox。
- 使用即時場地 keypoint 估計每個 frame 的 Homography。
- 將 player footpoint 與 Homography 整合到 BEV 投影流程，輸出左右並排分析影片。


## 執行階段提醒
請優先使用 **GPU** 或 **TPU** 的執行階段；不要使用純 CPU 執行。  
YOLO、MediaPipe 與影片處理在純 CPU 上會明顯較慢，容易讓課堂操作卡住。


## 課程流程
1. 選擇另一支參考影片。
2. 對每個 frame 同步執行 detection 與即時場地 keypoint。
3. 將即時場地 keypoint 與 Homography 整合後，把 player footpoint 持續投影到 BEV，產生 `d2_04_detector_keypoint_bev.mp4` 與對應 JSON。


In [ ]:
from pathlib import Path
import subprocess
import sys

COURSE_ROOT_HINT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "course_setup.py").exists()),
    Path("/content/basketball_hackathon/course"),
)
if not (COURSE_ROOT_HINT / "src" / "course_setup.py").exists() and "google.colab" in sys.modules:
    COURSE_ROOT_HINT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run([
        "git", "clone", "--depth", "1", "https://github.com/henry753951/basketball-hackathon-course.git", str(COURSE_ROOT_HINT)
    ], check=True)
if str(COURSE_ROOT_HINT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT_HINT))

from src.course_setup import bootstrap_course_repo  # noqa: E402

COURSE_ROOT = bootstrap_course_repo(COURSE_ROOT_HINT)


In [ ]:
from src.video_utils import display_video_in_notebook
from src.yolo_utils import write_detector_keypoint_bev_video

videos = sorted((COURSE_ROOT / "assets" / "raw" / "reference_videos").glob("*.mp4"))
if len(videos) < 2:
    raise FileNotFoundError("assets/raw/reference_videos/ 至少需要兩支參考影片。")

VIDEO_PATH = videos[1]
DETECTOR_PATH = COURSE_ROOT / "assets" / "models" / "detectors" / "yolo26n_basketball_player_best.pt"
COURT_MODEL_PATH = COURSE_ROOT / "assets" / "models" / "court_keypoints" / "yolo26n_basketball_court_pose_best.pt"
BEV_SPEC_PATH = COURSE_ROOT / "assets" / "samples" / "sample_bev_court.json"

print("video:", VIDEO_PATH)
print("detector:", DETECTOR_PATH)
print("court model:", COURT_MODEL_PATH)
print("start frame:", 90)


In [ ]:
bev_video = COURSE_ROOT / "assets" / "results" / "d2_04_detector_keypoint_bev.mp4"
bev_video, rows = write_detector_keypoint_bev_video(
    video_path=VIDEO_PATH,
    detector_path=DETECTOR_PATH,
    court_model_path=COURT_MODEL_PATH,
    bev_spec_path=BEV_SPEC_PATH,
    output_path=bev_video,
    max_frames=90,
    detector_conf=0.25,
    keypoint_conf=0.15,
    anchor_confidence=0.25,
    imgsz=960,
    start_frame=90,
)
bev_json = bev_video.with_suffix(".json")

print("BEV video:", bev_video)
print("BEV json:", bev_json)
print("projected rows:", len(rows))
display_video_in_notebook(bev_video, loop=True)


本單元輸出的 BEV 點位尚未維持跨 frame 身分；Day 3 會加入 ByteTrack，讓同一位球員在不同 frame 中保留穩定 track ID。

## 本單元產出檔案

- `assets/results/d2_04_detector_keypoint_bev.mp4`：detector + court keypoint + BEV 並排影片。
- `assets/results/d2_04_detector_keypoint_bev.json`：每個 frame 的 player footpoints 與 BEV 投影資料。
